In [ ]:
!pip install pandas

In [ ]:
!pip install matplotlib

In [ ]:
!pip install seaborn

In [ ]:
!pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
DATA_PATH = r"../Data/Raw/Kaggle_Triage_Modified.xlsx" 

In [ ]:
import os
print(os.getcwd())

In [ ]:
import pandas as pd

DATA_PATH = "../Data/Raw/Kaggle_Triage_Modified.xlsx"
df = pd.read_excel(DATA_PATH)

print("Shape:", df.shape)
print(df.head())

In [ ]:
df.info()

Target distribution

In [ ]:
print(df["disposition"].value_counts(dropna=False))
print("\nTarget proportions:")
print(df["disposition"].value_counts(normalize=True, dropna=False))

Encoded taget

In [ ]:
df = df.copy()

target_map = {
    "Admit": 1,
    "Discharge": 0
}

df["y"] = df["disposition"].map(target_map)

print(df["y"].isna().sum(), "rows have unmapped target")
df = df[df["y"].notna()].copy()
df["y"] = df["y"].astype(int)

print(df["y"].value_counts())

Check unique values of key categoricals

In [ ]:
for col in ["arrival_mode", "arrival_month", "arrival_day", "arrival_hour_bin"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))

Fix arrival_hour_bin corruption

In [ ]:
def clean_hour_bin(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    # direct expected values
    valid_bins = {"23-02", "03-06", "07-10", "11-14", "15-18", "19-22"}
    if x in valid_bins:
        return x
    
    # handle Excel / date-like corruption such as 2026-03-06 -> 03-06
    # or 2026-07-10 -> 07-10
    if len(x) >= 5 and x.count("-") >= 2:
        parts = x.split("-")
        if len(parts) >= 3:
            mm = parts[-2]
            dd = parts[-1]
            candidate = f"{mm}-{dd}"
            if candidate in valid_bins:
                return candidate
    
    return x  # keep original for inspection if unexpected

df["arrival_hour_bin_raw"] = df["arrival_hour_bin"]
df["arrival_hour_bin"] = df["arrival_hour_bin"].apply(clean_hour_bin)

print(df["arrival_hour_bin"].value_counts(dropna=False))

Normalize arrival_mode

In [ ]:
df["arrival_mode_missing"] = df["arrival_mode"].isna().astype(int)
df["arrival_mode"] = df["arrival_mode"].fillna("Unknown").astype(str).str.strip()

print(df["arrival_mode"].value_counts(dropna=False))

Numeric summary of vitals

In [ ]:
vital_cols = ["hr", "pulse_rate", "bp_systolic", "bp_diastole", "respiratory", "temperature", "spo2"]

df[vital_cols].describe().T

missingness (Indicator) flags for vitals

In [ ]:
for col in vital_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

missing_flag_cols = [f"{col}_missing" for col in vital_cols]
df["vitals_missing_count"] = df[missing_flag_cols].sum(axis=1)

df[missing_flag_cols + ["vitals_missing_count"]].head()

Compare admission rate by vital missingness

In [ ]:
for col in vital_cols:
    tmp = df.groupby(f"{col}_missing")["y"].mean().rename("admit_rate")
    print(f"\n{col}")
    print(tmp)

Reconcile hr and pulse_rate

In [ ]:
df["heart_rate_disagreement_flag"] = (
    df["hr"].notna() &
    df["pulse_rate"].notna() &
    ((df["hr"] - df["pulse_rate"]).abs() > 10)
).astype(int)

df["heart_rate_final"] = df["hr"].copy()
df.loc[df["heart_rate_final"].isna(), "heart_rate_final"] = df.loc[df["heart_rate_final"].isna(), "pulse_rate"]

df["heart_rate_final_missing"] = df["heart_rate_final"].isna().astype(int)

print("Disagreement rate:", df["heart_rate_disagreement_flag"].mean())
print("Final HR missing rate:", df["heart_rate_final_missing"].mean())

Create clinically interpretable physiologic flags

In [ ]:
df["tachycardia_flag"] = (df["heart_rate_final"] >= 120).astype(float)
df.loc[df["heart_rate_final"].isna(), "tachycardia_flag"] = np.nan

df["hypotension_flag"] = (df["bp_systolic"] < 90).astype(float)
df.loc[df["bp_systolic"].isna(), "hypotension_flag"] = np.nan

df["tachypnea_flag"] = (df["respiratory"] >= 24).astype(float)
df.loc[df["respiratory"].isna(), "tachypnea_flag"] = np.nan

df["hypoxia_flag"] = (df["spo2"] < 94).astype(float)
df.loc[df["spo2"].isna(), "hypoxia_flag"] = np.nan

df["fever_temp_flag"] = (df["temperature"] >= 100.4).astype(float)
df.loc[df["temperature"].isna(), "fever_temp_flag"] = np.nan

df["hypothermia_flag"] = (df["temperature"] < 95).astype(float)
df.loc[df["temperature"].isna(), "hypothermia_flag"] = np.nan

Binary column audit

In [ ]:
binary_cols = [
    "abdominal_pain", "chest_pain", "back_pain", "unclassified",
    "ckd", "dm", "headache", "htn", "cough", "chd", "asthma",
    "bronchitis", "dizziness", "glaucoma", "hepatitis", "paralysis",
    "pneumonia", "sicklecell", "arm_pain", "breast_pain", "dental_pain",
    "diarrhea", "ear_pain", "eye_pain", "fever", "pelvic_pain"
]

for col in binary_cols:
    vals = sorted(df[col].dropna().unique().tolist())
    if vals not in ([0], [1], [0,1]):
        print(col, vals)

Admission rate by key variables

In [ ]:
key_cols = [
    "arrival_mode", "arrival_hour_bin",
    "chest_pain", "abdominal_pain", "unclassified",
    "ckd", "dm", "htn", "chd", "pneumonia",
    "vitals_missing_count"
]

for col in key_cols:
    print(f"\n--- {col} ---")
    print(df.groupby(col)["y"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(20))

Save cleaned working dataset

In [ ]:
OUTPUT_PATH = "triage_cleaned_working.csv"
df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)